### MAPA DE MOVIMENTAÇÃO TRABALHISTA

#### Carrega as bibliotecas e as bases ####

In [0]:
pip install xlsxwriter

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
pip install pyxlsb 

  Obtaining dependency information for pyxlsb from https://files.pythonhosted.org/packages/7e/92/345823838ae367c59b63e03aef9c331f485370f9df6d049256a61a28f06d/pyxlsb-1.0.10-py2.py3-none-any.whl.metadata
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
pip install xlwings

  Obtaining dependency information for xlwings from https://files.pythonhosted.org/packages/23/0a/5b4f66c26a212f6cfd715fa2658d3ce7902499214ea638300e0984add9ce/xlwings-0.33.16-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.5 MB ? eta -:--:--
   ━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.2/1.5 MB 6.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 1.5/1.5 MB 23.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import os
import openpyxl 
import xlwings
import xlsxwriter
import shutil
import pyxlsb
import shutil
import numpy as np
import glob

### COMEÇA OS TRATAMENTOS ###

In [0]:
dbutils.widgets.text("Mes_Fechamento", "", "Base Fechamento Mensal")
Mes_Fechamento = dbutils.widgets.get("Mes_Fechamento")

In [0]:
meses_do_ano = [
    "JANEIRO", "FEVEREIRO", "MARÇO", "ABRIL", "MAIO", "JUNHO",
    "JULHO", "AGOSTO", "SETEMBRO", "OUTUBRO", "NOVEMBRO", "DEZEMBRO"
]

dbutils.widgets.dropdown(
    name="Mes_Extenso", 
    defaultValue="OUTUBRO", 
    choices=meses_do_ano, 
    label="Selecione o Mês por Extenso"
)

Meses = dbutils.widgets.get("Mes_Extenso")

In [0]:
#Carrega a base do fechamento do mês
base_fechamento_mensal = f'/Volumes/databox/juridico_comum/arquivos/MAPAS_MOVIMENTAÇÃO/INPUT/TRABALHISTA/Bases Fechamento Mensal/{Mes_Fechamento} Fechamento Trabalhista.xlsx'
#MAPA DE CALOR MES PASSADO PARA VALIDAÇÃO
mapa_movi_mes_passado = f'/Volumes/databox/juridico_comum/arquivos/MAPAS_MOVIMENTAÇÃO/INPUT/TRABALHISTA/MAPA_MES_ANTERIOR//Mapa de Movimentação  Trabalhista - {Meses} - 2025.xlsx'
#TEMPLATE DO MAPA DE MOVIMENTAÇÃO
template = f'/Volumes/databox/juridico_comum/arquivos/MAPAS_MOVIMENTAÇÃO/INPUT/TEMPLATE/Mapa de Movimentação Template.xlsx'

In [0]:
df_fechamento = pd.read_excel(
    base_fechamento_mensal,
    sheet_name= 'Base Fechamento',
    header=1
)

df_procv = pd.read_excel(
    base_fechamento_mensal,
    sheet_name= 'DEPARA_MAPA_MOVI',
    header=0
)

df_map_past = pd.read_excel(
    mapa_movi_mes_passado,
    sheet_name= 'Mapa de Movimentação Trab - Set',
    header=7
)

df_template = pd.read_excel(
    template,
    sheet_name= 'Mapa de Movimentação',
    header=7
)


#### Começa o tratamento da base para o realizar o mapa de movimentação (trabalhista é o unico que tem uma etapa a mais) ####

In [0]:
#CRIA A COLUNA CLASSIFICAÇÃO COM VALORES DE PROVAVEL OU REMOTO
cond_1 =  df_fechamento['Provisão Total (M)'] != 0
df_fechamento['Classificação'] = np.where(cond_1,'Provavel','remoto')

In [0]:
# Realziar o merge (procv) e traz a coluna com os nomes resumidos 

df_procv_res = df_procv[['EMPRESA (BASE FECHAMENTO)','EMPRESA (MAPA_NOVO)']]

df_fechamento_completo = pd.merge(
    df_fechamento,
    df_procv_res,
    left_on='Empresa (M)',
    right_on='EMPRESA (BASE FECHAMENTO)',
    how='left'
).drop('EMPRESA (BASE FECHAMENTO)',axis=1)

In [0]:
# Lista de empresas que possuem as variantes "ESPECIAL" e "MASSA"
empresas_especiais = ['BARTIRA', 'GLOBEX', 'GCBON', 'GCB']

# Definir as CONDIÇÕES para a mudança
conditions = [
    # Regra 1: Se for "COLETIVO" E ESTIVER NA LISTA
    (df_fechamento_completo['Área do Direito'] == 'TRABALHISTA COLETIVO') & 
    (df_fechamento_completo['EMPRESA (MAPA_NOVO)'].isin(empresas_especiais)),
    
    # Regra 2: Se for "TRABALHISTA" E ESTIVER NA LISTA
    (df_fechamento_completo['Área do Direito'] == 'TRABALHISTA') & 
    (df_fechamento_completo['EMPRESA (MAPA_NOVO)'].isin(empresas_especiais))
]

In [0]:
# Definir os RESULTADOS (escolhas) para cada condição
choices = [
    df_fechamento_completo['EMPRESA (MAPA_NOVO)'] + " ESPECIAL",    # Resultado Regra 1
    df_fechamento_completo['EMPRESA (MAPA_NOVO)'] + " MASSA"  # Resultado Regra 2
]

In [0]:
# Criamos uma nova coluna. 
df_fechamento_completo['EMPRESA_TEMPLATE'] = np.select(
    conditions, 
    choices, 
    default=df_fechamento_completo['EMPRESA (MAPA_NOVO)'] # O que fazer se NENHUMA condição bater
)

In [0]:
print("--- DataFrame Após Aplicar as Regras ---")
print(df_fechamento_completo[['Área do Direito', 'EMPRESA (MAPA_NOVO)', 'EMPRESA_TEMPLATE']], '\n')

--- DataFrame Após Aplicar as Regras ---
            Área do Direito EMPRESA (MAPA_NOVO) EMPRESA_TEMPLATE
0      TRABALHISTA COLETIVO                 GCB     GCB ESPECIAL
1      TRABALHISTA COLETIVO                 GCB     GCB ESPECIAL
2      TRABALHISTA COLETIVO                 GCB     GCB ESPECIAL
3      TRABALHISTA COLETIVO                 GCB     GCB ESPECIAL
4               TRABALHISTA                 GCB        GCB MASSA
...                     ...                 ...              ...
45480           TRABALHISTA                 GCB        GCB MASSA
45481           TRABALHISTA                 GCB        GCB MASSA
45482           TRABALHISTA                 GCB        GCB MASSA
45483           TRABALHISTA                 GCB        GCB MASSA
45484           TRABALHISTA                 GCB        GCB MASSA

[45485 rows x 3 columns] 



In [0]:
# 1. Cria o DF Remoto (Onde o valor é EXATAMENTE 0)
df_fechamento_completo_remoto = df_fechamento_completo[
    df_fechamento_completo['Provisão Total (M)'] == 0
].copy()

# 2. Cria o DF Provável (Onde o valor é DIFERENTE de 0)
df_fechamento_completo_provavel = df_fechamento_completo[
    df_fechamento_completo['Provisão Total (M)'] != 0
].copy()

In [0]:
# 1. Criar a tabela dinâmica APENAS para 'provavel'
df_provavel = df_fechamento_completo_provavel[df_fechamento_completo_provavel['Classificação'] == 'Provavel']

tab_provavel = pd.pivot_table(
    df_provavel, 
    index=['Área do Direito', 'EMPRESA_TEMPLATE'],
    values=['NOVOS','REATIVADOS','ESTOQUE'],
    aggfunc='sum',
    margins=False
)

In [0]:
# 2. Criar a tabela dinâmica APENAS para 'remoto'
df_remoto = df_fechamento_completo_remoto[df_fechamento_completo_remoto['Classificação'] == 'remoto']

tab_remoto = pd.pivot_table(
    df_remoto, 
    index=['Área do Direito', 'EMPRESA_TEMPLATE'],
    values=['NOVOS','REATIVADOS','ESTOQUE'],
    aggfunc='sum',
    margins=False
)

In [0]:
# Agora você tem duas tabelas: tab_provavel e tab_remoto
print("--- Tabela Provável ---")
print(tab_provavel)
print("\n--- Tabela Remoto ---")
print(tab_remoto)

--- Tabela Provável ---
                                       ESTOQUE  NOVOS  REATIVADOS
Área do Direito      EMPRESA_TEMPLATE                            
TRABALHISTA          ASAP                516.0    2.0         0.0
                     BARTIRA MASSA        95.0    0.0         0.0
                     CNT SOLUÇÕES          1.0    0.0         0.0
                     FUNDAÇÃO              1.0    0.0         0.0
                     GCB MASSA         21944.0   36.0        24.0
                     GCBHUB                4.0    0.0         0.0
                     GCBON MASSA          24.0    0.0         0.0
                     GLOBEX MASSA          3.0    0.0         0.0
TRABALHISTA COLETIVO ASAP                  1.0    0.0         0.0
                     GCB ESPECIAL        387.0    0.0         0.0

--- Tabela Remoto ---
                                       ESTOQUE  NOVOS  REATIVADOS
Área do Direito      EMPRESA_TEMPLATE                            
TRABALHISTA          ASAP    

### Monta o provavel ###

In [0]:
# Transforma as pivot em df para realizar o merge
df_provavel = tab_provavel.reset_index()
df_remoto = tab_remoto.reset_index()

In [0]:
# 1. Encontre a linha 'ASAP' que é 'TRABALHISTA COLETIVO'
condicao_asap = (df_provavel['EMPRESA_TEMPLATE'] == 'ASAP') & \
                (df_provavel['Área do Direito'] == 'TRABALHISTA COLETIVO')

# 2. Muda a 'Área do Direito' dessa linha para 'TRABALHISTA'
#    Agora, as linhas 0 e 8 têm as mesmas chaves: ('TRABALHISTA', 'ASAP')
df_provavel.loc[condicao_asap, 'Área do Direito'] = 'TRABALHISTA'

# 3. Agora, agrupe e some.
#    O groupby vai somar as linhas 0 e 8 automaticamente.
#    O GCB (linhas 4 e 9) não será afetado, pois suas chaves continuam diferentes.
df_provavel_agrupado = df_provavel.groupby(
    ['Área do Direito', 'EMPRESA_TEMPLATE']
).sum().reset_index()

df_provavel =  df_provavel_agrupado

In [0]:
df_provavel['NOVOS_TOTAL'] = df_provavel['NOVOS'] + df_provavel['REATIVADOS']

In [0]:
df_tp_fin = df_template[['EMPRESA']]

In [0]:
df_map_past.head()

,Unnamed: 0,EMPRESA,Saldo Inicial,Adição Novos,Encerrados,Saldo Final,Unnamed: 6,Provável Sdo inicial,Adição,Encerrados.1,Saldo Final.1,Unnamed: 11,Possivel Sdo inicial,Adição .1,Encerrados.2,Saldo Final.2,Unnamed: 16,Remoto Sdo inicial,Adição .2,Encerrados.3,Saldo Final.3,Unnamed: 21,Saldo Inicial.1,Adição Novo,Outras adições,Encerrado,Baixa parcial,Outras Reversões,Atualização juros,Saldo Final.4,Unnamed: 30,Saldo Inicial.2,Adição Novo.1,Outras adições.1,Encerrado.1,Baixa parcial.1,Outras Reversões.1,Atualização juros.1,Saldo Final.5,Unnamed: 39,Saldo Inicial.3,Adição Novo.2,Outras adições.2,Encerrado.2,Baixa parcial.2,Outras Reversões.2,Atualização juros.2,Saldo Final.6,Saldo Fechamento,Unnamed: 49,Unnamed: 50,Unnamed: 51
0,BANQI,BANQI,13.0,NaN,NaN,13.0,NaN,0.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,NaN,13.0,NaN,NaN,13.0,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,NaN,-3.209450e-01,NaN,NaN,NaN,NaN,NaN,NaN,-3.209450e-01,NaN,-3.209450e-01,0.0,0.0,0.00,0.0,0.000000e+00,0.000000,-3.209450e-01,NaN,0.32,0.0,0.0
1,OUTROS,NPC,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,0.0,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,NaN,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,0.000000e+00,0.0,0.0,0.00,0.0,0.000000e+00,0.000000,0.000000e+00,NaN,0.00,0.0,0.0
2,OUTROS,NPC ESPECIAL,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,0.0,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,NaN,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,0.000000e+00,0.0,0.0,0.00,0.0,0.000000e+00,0.000000,0.000000e+00,NaN,0.00,0.0,0.0
3,BARTIRA,BARTIRA MASSA,428.0,5.0,11.0,422.0,NaN,99.0,NaN,3.0,96.0,NaN,0.0,0.0,0.0,0.0,NaN,329.0,5.0,8.0,326.0,NaN,393125.179465,NaN,NaN,NaN,NaN,NaN,4066.231755,397191.41122,NaN,1.493945e+07,NaN,147062.5,-38949.71,NaN,-1.207742e+06,-86092.568197,1.375373e+07,NaN,1.533258e+07,0.0,147062.5,-38949.71,0.0,-1.207742e+06,-82026.336442,1.415092e+07,NaN,-14150923.44,0.0,0.0
4,BARTIRA,BARTIRA ESPECIAL,0.0,1.0,NaN,1.0,NaN,0.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,1.0,NaN,1.0,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,NaN,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,0.000000e+00,0.0,0.0,0.00,0.0,0.000000e+00,0.000000,0.000000e+00,NaN,NaN,0.0,0.0


In [0]:
#REALIZA O MERGE PARA TRAZER O SALDO FINAL DO MÊS ANTERIOR QUE SERÁ NOSSO MÊS INICIAL DO MÊS ATUAL

chave_merge = 'EMPRESA'
coluna = 'Saldo Final.1'
 
df_map_past= df_map_past.drop_duplicates(subset=[chave_merge])

df_mes_pass = df_map_past[[chave_merge, coluna]]

df_final = pd.merge(
    df_tp_fin,
    df_mes_pass,  
    on=chave_merge, 
    how='left'
)

In [0]:
df_final = df_final.rename(columns={'Saldo Final.1': 'Provável Sdo inicial'})

# 1 trocar nan por 0
df_final['Provável Sdo inicial'] = df_final['Provável Sdo inicial'].fillna(0)

# 2 Converter float para int
df_final['Provável Sdo inicial'] = df_final['Provável Sdo inicial'].astype(int)

df_final.head(20)

,EMPRESA,Provável Sdo inicial
0,BANQI,0
1,NPC,0
2,NPC ESPECIAL,0
3,BARTIRA MASSA,96
4,BARTIRA ESPECIAL,0
5,GLOBEX ESPECIAL,0
6,GLOBEX MASSA,3
7,GCBON ESPECIAL,0
8,GCBON MASSA,25
9,GCB ESPECIAL,186


In [0]:
chave_direita = 'EMPRESA_TEMPLATE' # Chave no df_provavel
chave_esquerda = 'EMPRESA'        # Chave no df_final
coluna_valor = 'NOVOS_TOTAL'

# 2. Criar o subset (isto está correto)
df_prov_merge = df_provavel[[chave_direita, coluna_valor]]

# 3. CORRIGIR O MERGE
df_final = pd.merge(
    df_final,
    df_prov_merge,
    left_on=chave_esquerda,     # Use 'EMPRESA' do df_final
    right_on=chave_direita,    # Use 'EMPRESA_TEMPLATE' do df_prov_merge
    how='left'
)

df_final = df_final.drop(chave_direita, axis=1)


In [0]:
df_final = df_final.rename(columns={'NOVOS_TOTAL': 'Novos'})

# 1 trocar nan por 0
df_final['Novos'] = df_final['Novos'].fillna(0)

# 2 Converter float para int
df_final['Novos'] = df_final['Novos'].astype(int)

df_drop = df_final.drop_duplicates(subset=['EMPRESA'], keep = 'first')

df_final = df_drop

In [0]:
chave_direita = 'EMPRESA_TEMPLATE' # Chave no df_provavel
chave_esquerda = 'EMPRESA'        # Chave no df_final
coluna_valor = 'ESTOQUE'

# 2. Criar o subset (isto está correto)
df_prov_merge1 = df_provavel[[chave_direita, coluna_valor]]

# 3. CORRIGIR O MERGE
df_final = pd.merge(
    df_final,
    df_prov_merge1,
    left_on=chave_esquerda,     # Use 'EMPRESA' do df_final
    right_on=chave_direita,    # Use 'EMPRESA_TEMPLATE' do df_prov_merge
    how='left'
)

df_final = df_final.drop(chave_direita, axis=1)

In [0]:
#df_final = df_final.rename(columns={'NOVOS_TOTAL': 'Adição'})

# 1 trocar nan por 0
df_final['ESTOQUE'] = df_final['ESTOQUE'].fillna(0)

# 2 Converter float para int
df_final['ESTOQUE'] = df_final['ESTOQUE'].astype(int)

In [0]:
# 1. CÁLCULO DO DELTA (A base de tudo)
#    Conta: (Saldo Inicial + Novos) - Estoque
#    Positivo = Sobra | Negativo = Falta | Zero = Bateu
delta = (
    df_final['Provável Sdo inicial'] + 
    df_final['Novos'] - 
    df_final['ESTOQUE']
)

# Apenas para facilitar, pegamos o valor absoluto (sem sinal de menos)
valor_absoluto = delta.abs().astype(int)

# 2. COLUNA 'ENCERRADO'
#    Regra: Se Delta > 0 (Sobrou item), registra a sobra aqui.
#    Caso contrário (Faltou ou Zero), coloca 0.
df_final['ENCERRADO'] = np.where(
    delta > 0, 
    valor_absoluto, 
    0
)

# 3. COLUNA 'resultado' (Sua coluna de Adição)
#    Regra A: Se Delta < 0 (Faltou item), coloca a DIFERENÇA (valor calculado).
#    Regra B: Se Delta >= 0 (Sobrou ou Zero), MANTÉM OS NOVOS ORIGINAIS.
df_final['resultado'] = np.where(
    delta < 0, 
    valor_absoluto,           # Caso GCB Especial (Substitui 11 por 94)
    df_final['Novos']  # Caso BANQI e GCBHUB (Mantém o 1 e o 2)
)

In [0]:
#df_final = df_final.drop(['ESTOQUE'], axis=1)
#df_final = df_final.drop(['Novos'], axis=1)

In [0]:
df_final.head(20)

,EMPRESA,Provável Sdo inicial,Novos,ESTOQUE,ENCERRADO,resultado
0,BANQI,0,0,0,0,0
1,NPC,0,0,0,0,0
2,NPC ESPECIAL,0,0,0,0,0
3,BARTIRA MASSA,96,0,95,1,0
4,BARTIRA ESPECIAL,0,0,0,0,0
5,GLOBEX ESPECIAL,0,0,0,0,0
6,GLOBEX MASSA,3,0,3,0,0
7,GCBON ESPECIAL,0,0,0,0,0
8,GCBON MASSA,25,0,24,1,0
9,GCB ESPECIAL,186,0,387,0,201


In [0]:
df_final = df_final.rename(columns={'resultado': 'Adição'})

In [0]:
df_final = df_final.drop(['ESTOQUE'], axis=1)
df_final = df_final.drop(['Novos'], axis=1)

In [0]:
df_final.head()

,EMPRESA,Provável Sdo inicial,ENCERRADO,Adição
0,BANQI,0,0,0
1,NPC,0,0,0
2,NPC ESPECIAL,0,0,0
3,BARTIRA MASSA,96,1,0
4,BARTIRA ESPECIAL,0,0,0


### Monta o remoto ###

In [0]:
# 1. Encontre a linha 'ASAP' que é 'TRABALHISTA COLETIVO'
condicao_asap = (df_remoto['EMPRESA_TEMPLATE'] == 'ASAP') & \
                (df_remoto['Área do Direito'] == 'TRABALHISTA COLETIVO')

# 2. Mude a 'Área do Direito' dessa linha para 'TRABALHISTA'
#    Agora, as linhas 0 e 8 têm as mesmas chaves: ('TRABALHISTA', 'ASAP')
df_remoto.loc[condicao_asap, 'Área do Direito'] = 'TRABALHISTA'

# 3.
#    O groupby vai somar as linhas 0 e 8 automaticamente.
#    O GCB (linhas 4 e 9) não será afetado, pois suas chaves continuam diferentes.
df_remoto_agrupado = df_remoto.groupby(
    ['Área do Direito', 'EMPRESA_TEMPLATE']
).sum().reset_index()

df_remoto =  df_remoto_agrupado

In [0]:
# 1. Encontre TODAS as linhas que são 'TRABALHISTA COLETIVO'
condicao_coletivo = (df_remoto['Área do Direito'] == 'TRABALHISTA COLETIVO')

# 2. Mude a 'Área do Direito' dessas linhas para 'TRABALHISTA'
#    Isso vai fazer com que ASAP, GCBHUB, etc., fiquem com a mesma chave.
df_remoto.loc[condicao_coletivo, 'Área do Direito'] = 'TRABALHISTA'

# 3. Agora, agrupe e some pelas chaves
#    Isso irá somar automaticamente ASAP, GCBHUB e qualquer outra empresa.
df_remoto_agrupado = df_remoto.groupby(
    ['Área do Direito', 'EMPRESA_TEMPLATE']
).sum().reset_index()

df_remoto =  df_remoto_agrupado

In [0]:
# Soma os valores novos com os reativados
df_remoto['NOVOS_TOTAL'] = df_remoto['NOVOS'] + df_remoto['REATIVADOS']

In [0]:
# Separa a coluna EMPRESA para criar o novo dataframe 
df_tp_fin1 = df_template[['EMPRESA']]

In [0]:
#Realiza o merge para trazer o valor final do mes anterior para se tornar o mes atual
chave_merge_remoto = 'EMPRESA'
coluna = 'Saldo Final.3'
 
df_map_past1= df_map_past.drop_duplicates(subset=[chave_merge_remoto])

df_mes_pass_remoto = df_map_past1[[chave_merge_remoto, coluna]]

df_final_remoto = pd.merge(
    df_tp_fin,
    df_mes_pass_remoto,  
    on=chave_merge, 
    how='left'
)

In [0]:
df_final_remoto = df_final_remoto.rename(columns={'Saldo Final.3': 'Remoto Sdo inicial'})

# 1 trocar nan por 0
df_final_remoto['Remoto Sdo inicial'] = df_final_remoto['Remoto Sdo inicial'].fillna(0)

# 2 Converter float para int
df_final_remoto['Remoto Sdo inicial'] = df_final_remoto['Remoto Sdo inicial'].astype(int)

In [0]:
chave_direita = 'EMPRESA_TEMPLATE' # Chave no df_provavel
chave_esquerda = 'EMPRESA'        # Chave no df_final
coluna_valor = 'NOVOS_TOTAL'

# 2. Criar o subset (isto está correto)
df_remoto_merge = df_remoto[[chave_direita, coluna_valor]]

# 3. CORRIGIR O MERGE
df_final_remoto = pd.merge(
    df_final_remoto,
    df_remoto_merge,
    left_on=chave_esquerda,     # Use 'EMPRESA' do df_final
    right_on=chave_direita,    # Use 'EMPRESA_TEMPLATE' do df_prov_merge
    how='left'
)

df_final_remoto = df_final_remoto.drop(chave_direita, axis=1)

In [0]:
#Converte os numeros de float para int
df_final_remoto = df_final_remoto.rename(columns={'NOVOS_TOTAL': 'Novos'})

# 1 trocar nan por 0
df_final_remoto['Novos'] = df_final_remoto['Novos'].fillna(0)

# 2 Converter float para int
df_final_remoto['Novos'] = df_final_remoto['Novos'].astype(int)

df_drop_remoto = df_final_remoto.drop_duplicates(subset=['EMPRESA'], keep = 'first')

df_final_remoto = df_drop_remoto

In [0]:
chave_direita = 'EMPRESA_TEMPLATE' # Chave no df_provavel
chave_esquerda = 'EMPRESA'        # Chave no df_final
coluna_valor = 'ESTOQUE'

# 2. Criar o subset (isto está correto)
df_remoto_merge1 = df_remoto[[chave_direita, coluna_valor]]

# 3. CORRIGIR O MERGE
df_final_remoto = pd.merge(
    df_final_remoto,
    df_remoto_merge1,
    left_on=chave_esquerda,     # Use 'EMPRESA' do df_final
    right_on=chave_direita,    # Use 'EMPRESA_TEMPLATE' do df_prov_merge
    how='left'
)

df_final_remoto = df_final_remoto.drop(chave_direita, axis=1)

In [0]:
# Converte float para int 

# 1 trocar nan por 0
df_final_remoto['ESTOQUE'] = df_final_remoto['ESTOQUE'].fillna(0)

# 2 Converter float para int
df_final_remoto['ESTOQUE'] = df_final_remoto['ESTOQUE'].astype(int)

In [0]:
# 1. CÁLCULO DO DELTA (A base de tudo)
#    Conta: (Saldo Inicial + Novos) - Estoque
#    Positivo = Sobra | Negativo = Falta | Zero = Bateu
delta = (
    df_final_remoto['Remoto Sdo inicial'] + 
    df_final_remoto['Novos'] - 
    df_final_remoto['ESTOQUE']
)

# Apenas para facilitar, pegamos o valor absoluto (sem sinal de menos)
valor_absoluto = delta.abs().astype(int)

# 2. COLUNA 'ENCERRADO'
#    Regra: Se Delta > 0 (Sobrou item), registra a sobra aqui.
#    Caso contrário (Faltou ou Zero), coloca 0.
df_final_remoto['ENCERRADO'] = np.where(
    delta > 0, 
    valor_absoluto, 
    0
)

# 3. COLUNA 'resultado' (Sua coluna de Adição)
#    Regra A: Se Delta < 0 (Faltou item), coloca a DIFERENÇA (valor calculado).
#    Regra B: Se Delta >= 0 (Sobrou ou Zero), MANTÉM OS NOVOS ORIGINAIS.
df_final_remoto['resultado'] = np.where(
    delta < 0, 
    valor_absoluto,           # Caso GCB Especial (Substitui 11 por 94)
    df_final_remoto['Novos']  # Caso BANQI e GCBHUB (Mantém o 1 e o 2)
)

In [0]:
df_final_remoto.head(20)

,EMPRESA,Remoto Sdo inicial,Novos,ESTOQUE,ENCERRADO,resultado
0,BANQI,13,1,14,0,1
1,NPC,0,0,0,0,0
2,NPC ESPECIAL,0,0,0,0,0
3,BARTIRA MASSA,326,5,330,1,5
4,BARTIRA ESPECIAL,1,0,1,0,0
5,GLOBEX ESPECIAL,1,0,1,0,0
6,GLOBEX MASSA,8,0,6,2,0
7,GCBON ESPECIAL,0,0,0,0,0
8,GCBON MASSA,14,0,15,0,1
9,GCB ESPECIAL,283,11,388,0,94


In [0]:
df_final_remoto = df_final_remoto.rename(columns={'resultado': 'Adição'})

In [0]:
df_final_remoto = df_final_remoto.drop(['ESTOQUE'], axis=1)
df_final_remoto = df_final_remoto.drop(['Novos'], axis=1)

In [0]:
df_final_remoto.head(20)

,EMPRESA,Remoto Sdo inicial,ENCERRADO,Adição
0,BANQI,13,0,1
1,NPC,0,0,0
2,NPC ESPECIAL,0,0,0
3,BARTIRA MASSA,326,1,5
4,BARTIRA ESPECIAL,1,0,0
5,GLOBEX ESPECIAL,1,0,0
6,GLOBEX MASSA,8,2,0
7,GCBON ESPECIAL,0,0,0
8,GCBON MASSA,14,0,1
9,GCB ESPECIAL,283,0,94


## Realiza a valorização por empresa ##

In [0]:
caminho_da_pasta = f'/Volumes/databox/juridico_comum/arquivos/MAPAS_MOVIMENTAÇÃO/INPUT/EMPRESAS/'
extensao_dos_arquivos = 'xlsx'
coluna_para_buscar = 'Centro de Custo SAP'
texto_para_achar = 'Saldo Total'
linhas_de_cabecalho_possiveis = [8, 9, 10, 13]
nome_da_nova_coluna = 'Empresa_NOME'
posicao_palavra_empresa = [2,3]
EMPRESAS_SEM_MASSA_OU_ESTRATEGICO = ['INTEGRA', 'CNTLOG', 'BANQI', 'VVLOG', 'VIAHUB', 'CELER', 'CNT','FUNDAÇÃO']

In [0]:
lista_dfs_filtrados = []

# Encontrar todos os arquivos com a extensão correta na pasta
padrao_busca = os.path.join(caminho_da_pasta, f'*.{extensao_dos_arquivos}')
lista_de_arquivos = glob.glob(padrao_busca)

if not lista_de_arquivos:
    print(f"Aviso: Nenhum arquivo com a extensão '.{extensao_dos_arquivos}' foi encontrado em:")
    print(caminho_da_pasta)
else:
    print(f"Encontrados {len(lista_de_arquivos)} arquivos. Processando...")

# Loop principal: processa um arquivo de cada vez
for arquivo in lista_de_arquivos:
    nome_base_arquivo = os.path.basename(arquivo)
    print(f"   -> Processando arquivo: {nome_base_arquivo}")
    
    # --- [INÍCIO DA LÓGICA CORRIGIDA] ---
    # Lógica para extrair o nome da empresa
    nome_da_empresa = "NomeNaoEncontrado" # Valor padrão se falhar
    try:
        nome_sem_extensao = os.path.splitext(nome_base_arquivo)[0]
        palavras = nome_sem_extensao.split(' ') 
        
        # Pega os índices da sua lista [2, 3]
        idx1 = posicao_palavra_empresa[0] # Pega o índice 2
        idx2 = posicao_palavra_empresa[1] # Pega o índice 3
        
        # Verifica se a lista de palavras tem pelo menos a primeira palavra (idx1)
        if len(palavras) > idx1: 
            palavra1 = palavras[idx1] # Ex: "BARTIRA" ou "INTEGRA"

            # *** NOVA LÓGICA DE VERIFICAÇÃO ***
            # Verifica se a 'palavra1' está na lista de exceções
            if palavra1 in EMPRESAS_SEM_MASSA_OU_ESTRATEGICO:
                nome_da_empresa = palavra1
                print(f"      Nome (Exceção) extraído: '{nome_da_empresa}'")
            
            # Se NÃO for exceção, tenta pegar a segunda palavra (lógica antiga)
            elif len(palavras) > idx2: 
                palavra2 = palavras[idx2] # Ex: "MASSA"
                nome_da_empresa = f"{palavra1} {palavra2}" # Ex: "BARTIRA MASSA"
                print(f"      Nome (Padrão) extraído: '{nome_da_empresa}'")
            
            else:
                # Se não é exceção E não tem a palavra 2, algo está errado
                print(f"      AVISO: Nome '{nome_base_arquivo}' não tem 4ª palavra (e a 3ª não é exceção). Usando só a 3ª: '{palavra1}'")
                nome_da_empresa = palavra1 # Usa a palavra 1 como último recurso

        else:
            # Se nem a palavra 1 existe
            print(f"      AVISO: Nome do arquivo '{nome_base_arquivo}' é muito curto para extrair a 3ª palavra.")
            
    except Exception as e:
        print(f"      AVISO: Erro ao extrair nome da empresa do arquivo: {e}")
    # --- [FIM DA LÓGICA CORRIGIDA] ---

    try:
        # --- (LÓGICA DE LEITURA) ---
        df_temp = None
        cabecalho_encontrado = False
        
        # 1. Tenta ler com os cabeçalhos definidos
        for linha_cabecalho in linhas_de_cabecalho_possiveis: # Tenta [8, 10, 13]
            df_lido_temp = None
            try:
                # Tenta ler o arquivo
                if extensao_dos_arquivos == 'csv':
                    df_lido_temp = pd.read_csv(arquivo, header=linha_cabecalho)
                elif extensao_dos_arquivos in ['xls', 'xlsx']:
                    df_lido_temp = pd.read_excel(arquivo, header=linha_cabecalho)
                elif extensao_dos_arquivos == 'xlsb':
                    df_lido_temp = pd.read_excel(arquivo, engine='pyxlsb', header=linha_cabecalho)
                
                # Se leu, verifica se a coluna-chave está presente
                if df_lido_temp is not None and coluna_para_buscar in df_lido_temp.columns:
                    df_temp = df_lido_temp # Sucesso! Este é o DF correto.
                    cabecalho_encontrado = True
                    print(f"      Cabeçalho correto encontrado na linha {linha_cabecalho + 1}.")
                    break # Para o loop interno, pois já achamos
                else:
                    # Se leu mas não achou a coluna, apenas informa (não é um erro ainda)
                    print(f"      Info: Cabeçalho na linha {linha_cabecalho + 1} não contém '{coluna_para_buscar}'. Tentando próximo...")
                    
            except Exception as e_leitura:
                # Captura erros de leitura (ex: arquivo vazio, formato inválido)
                print(f"      AVISO: Falha ao tentar ler com cabeçalho na linha {linha_cabecalho + 1}. {e_leitura}")
                # Continua para a próxima tentativa de cabeçalho
                pass 
        
        # 2. Verifica se algum cabeçalho funcionou
        if not cabecalho_encontrado:
            print(f"      ERRO: A coluna '{coluna_para_buscar}' não foi encontrada com nenhum dos cabeçalhos testados ({[h+1 for h in linhas_de_cabecalho_possiveis]}). Pulando arquivo.")
            continue # Pula para o próximo ARQUIVO
        # --- (FIM DA NOVA LÓGICA DE LEITURA) ---

        # Se chegamos aqui, df_temp é VÁLIDO.
        
        # Filtro (como antes)
        df_filtrado = df_temp[df_temp[coluna_para_buscar].str.contains(texto_para_achar, case=False, na=False)].copy()

        # --- Adiciona a coluna da empresa ---
        if not df_filtrado.empty:
            print(f"      Encontradas {len(df_filtrado)} linhas com '{texto_para_achar}'.")
            
            # Use .insert() para adicionar a coluna na posição 0 (Coluna A)
            df_filtrado.insert(loc=0, 
                                column=nome_da_nova_coluna, 
                                value=nome_da_empresa)
            
            # Adiciona o DF (agora com a nova coluna) na lista
            lista_dfs_filtrados.append(df_filtrado)
        else:
            print(f"      Nenhuma linha encontrada neste arquivo.")
            
    except Exception as e:
        # Pega-tudo para erros gerais (como o filtro falhar, etc.)
        print(f"      ERRO GERAL ao processar {nome_base_arquivo}: {e}. Pulando.")


# --- Passo Final: Juntar tudo ---
if lista_dfs_filtrados:
    # Concatena todos os DFs (que agora já têm a coluna "Empresa")
    df_final_combinado = pd.concat(lista_dfs_filtrados, ignore_index=True)
    
    print("\n--- SUCESSO! ---")
    print("DataFrame final combinado:")
    print(df_final_combinado)
    
    nome_arquivo_saida = 'resultado_combinado_saldo_total.csv'
    df_final_combinado.to_csv(nome_arquivo_saida, index=False)
    print(f"\nResultado salvo em '{nome_arquivo_saida}'")
    
else:
    print("\nProcessamento concluído. Nenhuma linha com 'saldo total' foi encontrada em nenhum dos arquivos.")

Encontrados 13 arquivos. Processando...
   -> Processando arquivo: Fechamento TRABALHISTA ASAP LOG MASSA _ .xlsx
      Nome (Padrão) extraído: 'ASAP LOG'
      Cabeçalho correto encontrado na linha 9.
      Encontradas 1 linhas com 'Saldo Total'.
   -> Processando arquivo: Fechamento TRABALHISTA ASAP LOG MASSA.xlsx
      Nome (Padrão) extraído: 'ASAP LOG'
      Cabeçalho correto encontrado na linha 9.
      Encontradas 1 linhas com 'Saldo Total'.
   -> Processando arquivo: Fechamento TRABALHISTA BANQI.xlsx
      Nome (Exceção) extraído: 'BANQI'
      Cabeçalho correto encontrado na linha 9.
      Encontradas 1 linhas com 'Saldo Total'.
   -> Processando arquivo: Fechamento TRABALHISTA BARTIRA MASSA.xlsx
      Nome (Padrão) extraído: 'BARTIRA MASSA'
      Cabeçalho correto encontrado na linha 9.
      Encontradas 1 linhas com 'Saldo Total'.
   -> Processando arquivo: Fechamento TRABALHISTA CELER.xlsx
      Nome (Exceção) extraído: 'CELER'
      Cabeçalho correto encontrado na linha 9.
 

In [0]:
mapa_empr = {'VIAHUB' : 'GCBHUB'}
df_final_combinado['Empresa_NOME'] = df_final_combinado['Empresa_NOME'].replace(mapa_empr)


In [0]:
df_final_combinado.head(25)

,Empresa_NOME,Unnamed: 0,Pasta,Centro de Custo SAP,C/ S/ PGTO,RNO,Empresa,Empresa.,Saldo Inicial M-1 SOCIO+EMPRESA,Saldo Inicial SÓCIO,ADIÇÃO NOVO,Qnt,OUTRAS ADIÇÕES,Qnt.1,ENCERRADO,Qnt.2,OUTRAS REVERSÕES,Qnt.3,ATUALIZAÇÃO,Qnt.4,Saldo FINAL SÓCIO,Unnamed: 20,Saldo Inicial EMPRESA,ADIÇÃO NOVO.1,Qnt.5,OUTRAS ADIÇÕES.1,Qnt.6,OUTRAS ADIÇÕES ENTRE EMPRESAS,Qnt.7,ENCERRADO.1,Qnt.8,OUTRAS REVERSÕES.1,Qnt.9,OUTRAS REVERSÕES ENTRE EMPRESAS,Qnt.10,ATUALIZAÇÃO.1,Qnt.11,Saldo FINAL EMPRESA,Unnamed: 37,Saldo Final TOTAL SÓCIO + EMPRESA,Unnamed: 24,OUTRAS ADIÇÕES ENTRE EMPRESAS.1,OUTRAS REVERSÕES ENTRE EMPRESAS.1,Qnt.12,Qnt.13,Unnamed: 41,Saldo incial Sócio,Saldo FINAL Sócio,Unnamed: 18,Saldo incial Empresa,Saldo final Empresa,Unnamed: 33
0,ASAP LOG,NaN,NaN,Saldo Total,NaN,NaN,NaN,NaN,41371098.464858,0,0.0,NaN,0.0,NaN,0.0,NaN,0.000000e+00,NaN,0.000000,NaN,0.000000e+00,NaN,4.137110e+07,3.816597e+04,NaN,1.979781e+06,NaN,8614.471359,NaN,-2.745729e+05,NaN,-1.297894e+06,NaN,0.0,NaN,3.077389e+05,NaN,4.213293e+07,NaN,4.213293e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ASAP LOG,NaN,NaN,Saldo Total,NaN,NaN,NaN,NaN,119775.355272,0,0.0,NaN,0.0,NaN,0.0,NaN,0.000000e+00,NaN,0.000000,NaN,0.000000e+00,NaN,1.197754e+05,0.000000e+00,NaN,0.000000e+00,NaN,0.000000,NaN,0.000000e+00,NaN,-4.818871e+04,NaN,0.0,NaN,-8.195558e+03,NaN,6.339109e+04,NaN,6.339109e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BANQI,NaN,NaN,Saldo Total,NaN,NaN,NaN,NaN,0,0,0.0,NaN,0.0,NaN,0.0,NaN,0.000000e+00,NaN,0.000000,NaN,0.000000e+00,NaN,0.000000e+00,0.000000e+00,NaN,0.000000e+00,NaN,0.000000,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.0,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BARTIRA MASSA,NaN,NaN,Saldo Total,NaN,NaN,NaN,NaN,14150923.427333,397191.41127,0.0,NaN,0.0,NaN,0.0,NaN,0.000000e+00,NaN,4266.210366,NaN,4.014576e+05,NaN,1.375373e+07,0.000000e+00,NaN,1.738875e+05,NaN,0.000000,NaN,-7.919978e+04,NaN,-2.355797e+05,NaN,0.0,NaN,2.646484e+05,NaN,1.387749e+07,NaN,1.427895e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CELER,NaN,NaN,Saldo Total,NaN,NaN,NaN,NaN,0,0,0.0,NaN,0.0,NaN,0.0,NaN,0.000000e+00,NaN,0.000000,NaN,0.000000e+00,NaN,0.000000e+00,0.000000e+00,NaN,0.000000e+00,NaN,0.000000,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.0,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CNT,NaN,NaN,Saldo Total,NaN,NaN,NaN,NaN,10569.247776,0,0.0,NaN,0.0,NaN,0.0,NaN,0.000000e+00,NaN,0.000000,NaN,0.000000e+00,NaN,1.056925e+04,0.000000e+00,NaN,0.000000e+00,NaN,0.000000,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.0,NaN,1.253116e+02,NaN,1.069456e+04,NaN,1.069456e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,FUNDAÇÃO,NaN,NaN,Saldo Total,NaN,NaN,NaN,NaN,654682.623445,0,0.0,NaN,0.0,NaN,0.0,NaN,0.000000e+00,NaN,0.000000,NaN,0.000000e+00,NaN,6.546826e+05,0.000000e+00,NaN,0.000000e+00,NaN,0.000000,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.0,NaN,7.492121e+03,NaN,6.621747e+05,NaN,6.621747e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,GCB ESPECIAL,NaN,NaN,Saldo Total,NaN,NaN,NaN,NaN,17041089.665185,293683.481942,0,NaN,0,NaN,0,NaN,0.000000e+00,NaN,2350.221780,NaN,2.960337e+05,NaN,1.674741e+07,0.000000e+00,NaN,5.003784e+05,NaN,0.000000,NaN,-1.264084e+05,NaN,0.000000e+00,NaN,0.0,NaN,1.353995e+05,NaN,1.725678e+07,NaN,1.755281e+07,NaN,0.0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,GCB MASSA,NaN,NaN,Saldo Total,NaN,NaN,NaN,NaN,1602175453.023418,49112149.969335,71086.302941,NaN,403350.939223,NaN,-79913.746315,NaN,-2.640340e+06,NaN,296755.048280,NaN,4.716309e+07,NaN,1.553063e+09,2.070956e+06,NaN,3.101674e+07,NaN,0.000000,NaN,-1.021341e+07,NaN,-2.857420e+07,NaN,0.0,NaN,1.324901e+07,NaN,1.560604e+09,NaN,1.607767e+09,NaN,0.0,-8614.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,GCBON MASSA,NaN,NaN,Saldo Total,NaN,NaN,NaN,NaN,6612370.31832,NaN,0,NaN,0.0,NaN,0.0,NaN,-3.026842e+01,NaN,-19819.649886,NaN,NaN,NaN,NaN,0.000000e+00,NaN,0.000000e+00,NaN,NaN,NaN,0.000000e+00,NaN,-9.705948e+05,N

In [0]:
colunas_drop = ['Unnamed: 0', 'Pasta', 'Centro de Custo SAP', 'C/ S/ PGTO', 'RNO', 'Empresa', 'Empresa.', 'Qnt', 'Qnt.1', 'Qnt.2', 'Qnt.3', 'Qnt.4', 'Qnt.5', 'Qnt.6', 'Qnt.7', 'Qnt.8', 'Qnt.9', 'Qnt.10', 'Qnt.11', 'Unnamed: 37', 'Saldo Final TOTAL SÓCIO + EMPRESA', 'Unnamed: 24', 'OUTRAS ADIÇÕES ENTRE EMPRESAS.1', 'OUTRAS REVERSÕES ENTRE EMPRESAS.1', 'Qnt.12', 'Qnt.13', 'Unnamed: 41', 'Saldo FINAL Sócio', 'Unnamed: 18', 'Saldo final Empresa', 'Unnamed: 33','Saldo Inicial M-1 SOCIO+EMPRESA']

In [0]:
df_final_combinado.drop(colunas_drop, axis=1, inplace=True)

In [0]:
# 1. Liste todas as colunas que você quer que sejam somadas
colunas_para_somar = [
    'Saldo Inicial M-1 SOCIO+EMPRESA', 
    'Saldo Inicial SÓCIO', 
    'ADIÇÃO NOVO',
    'OUTRAS ADIÇÕES',
    'ENCERRADO',
    'OUTRAS REVERSÕES',
    'ATUALIZAÇÃO',
    'Saldo FINAL SÓCIO',
    'Unnamed: 20',
    'Saldo Inicial EMPRESA',
    'ADIÇÃO NOVO.1',
    'OUTRAS ADIÇÕES.1',
    'OUTRAS ADIÇÕES ENTRE EMPRESAS',
    'ENCERRADO.1',
    'OUTRAS REVERSÕES.1',
    'OUTRAS REVERSÕES ENTRE EMPRESAS',
    'ATUALIZAÇÃO.1',
    'Saldo FINAL EMPRESA'
]

# 2. Faça um loop para forçar a conversão de cada uma
for col in colunas_para_somar:
    # Verifica se a coluna realmente existe no DataFrame antes de tentar
    if col in df_final_combinado.columns:
        
        # Converte para numérico. 'errors='coerce'' transforma 
        # qualquer texto/erro em NaN (Nulo).
        df_final_combinado[col] = pd.to_numeric(
            df_final_combinado[col], errors='coerce'
        )
        
        # Substitui todos os NaNs (originais ou criados agora) por 0
        df_final_combinado[col] = df_final_combinado[col].fillna(0)

print("Conversão para numérico concluída.")

# 3. AGORA seu código original do bloco 52 vai funcionar
df_agrupado = df_final_combinado.groupby('Empresa_NOME').sum().reset_index()

# Verifique o resultado
print("\n--- DataFrame Agrupado Corretamente ---")
display(df_agrupado.head())

Conversão para numérico concluída.

--- DataFrame Agrupado Corretamente ---


/home/spark-31004327-d8ac-451b-8c5d-fc/.ipykernel/2991/command-6683834362431664-3016427825:40: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df_agrupado = df_final_combinado.groupby('Empresa_NOME').sum().reset_index()


Empresa_NOME,Saldo Inicial SÓCIO,ADIÇÃO NOVO,OUTRAS ADIÇÕES,ENCERRADO,OUTRAS REVERSÕES,ATUALIZAÇÃO,Saldo FINAL SÓCIO,Unnamed: 20,Saldo Inicial EMPRESA,ADIÇÃO NOVO.1,OUTRAS ADIÇÕES.1,OUTRAS ADIÇÕES ENTRE EMPRESAS,ENCERRADO.1,OUTRAS REVERSÕES.1,OUTRAS REVERSÕES ENTRE EMPRESAS,ATUALIZAÇÃO.1,Saldo FINAL EMPRESA,Saldo incial Empresa
ASAP LOG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.1490873820129946E7,38165.9677378431,1979780.623949712,8614.47135881483,-274572.8545729731,-1346082.9525313345,0.0,299543.3056636266,4.2196322381735645E7,0.0
BANQI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BARTIRA MASSA,397191.4112702489,0.0,0.0,0.0,0.0,4266.210366217614,401457.6216364666,0.0,1.3753732016062753E7,0.0,173887.53,0.0,-79199.78,-235579.7050489163,0.0,264648.4361276703,1.3877488497141505E7,0.0
CELER,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CNT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10569.2477761812,0.0,0.0,0.0,0.0,0.0,0.0,125.31157052160017,10694.559346702801,0.0


In [0]:
# 1. Crie o dicionário com as mudanças
mapa_nomes = {
    'ASAP LOG': 'ASAP',
    'CELER': 'CELER PROCESSAMENTO',
    'CNT': 'CNT SOLUÇÕES'
}

# 2. Use .replace() na coluna 'Empresa_NOME'
df_agrupado['Empresa_NOME'] = df_agrupado['Empresa_NOME'].replace(mapa_nomes)

# 3. Agora verifique o resultado
print(df_agrupado.head())

          Empresa_NOME  ...  Saldo incial Empresa
0                 ASAP  ...                   0.0
1                BANQI  ...                   0.0
2        BARTIRA MASSA  ...                   0.0
3  CELER PROCESSAMENTO  ...                   0.0
4         CNT SOLUÇÕES  ...                   0.0

[5 rows x 19 columns]


In [0]:
df = pd.merge(
    df_final,
    df_final_remoto,
    on='EMPRESA',
    how='left'
)

df.head(20)

,EMPRESA,Provável Sdo inicial,ENCERRADO_x,Adição_x,Remoto Sdo inicial,ENCERRADO_y,Adição_y
0,BANQI,0,0,0,13,0,1
1,NPC,0,0,0,0,0,0
2,NPC ESPECIAL,0,0,0,0,0,0
3,BARTIRA MASSA,96,1,0,326,1,5
4,BARTIRA ESPECIAL,0,0,0,1,0,0
5,GLOBEX ESPECIAL,0,0,0,1,0,0
6,GLOBEX MASSA,3,0,0,8,2,0
7,GCBON ESPECIAL,0,0,0,0,0,0
8,GCBON MASSA,25,1,0,14,0,1
9,GCB ESPECIAL,186,0,201,283,0,94


In [0]:
# Use 'left_on' e 'right_on' para especificar as colunas-chave

df_resultado = pd.merge(
    df,                     # DataFrame da Esquerda
    df_agrupado,            # DataFrame da Direita
    left_on='EMPRESA',      # Chave do DataFrame da esquerda (df)
    right_on='Empresa_NOME',# Chave do DataFrame da direita (df_agrupado)
    how='left'
)

# Visualizar o resultado
df_resultado.head()

,EMPRESA,Provável Sdo inicial,ENCERRADO_x,Adição_x,Remoto Sdo inicial,ENCERRADO_y,Adição_y,Empresa_NOME,Saldo Inicial SÓCIO,ADIÇÃO NOVO,OUTRAS ADIÇÕES,ENCERRADO,OUTRAS REVERSÕES,ATUALIZAÇÃO,Saldo FINAL SÓCIO,Unnamed: 20,Saldo Inicial EMPRESA,ADIÇÃO NOVO.1,OUTRAS ADIÇÕES.1,OUTRAS ADIÇÕES ENTRE EMPRESAS,ENCERRADO.1,OUTRAS REVERSÕES.1,OUTRAS REVERSÕES ENTRE EMPRESAS,ATUALIZAÇÃO.1,Saldo FINAL EMPRESA,Saldo incial Empresa
0,BANQI,0,0,0,13,0,1,BANQI,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000e+00,0.0,0.00,0.0,0.00,0.000000,0.0,0.000000,0.000000e+00,0.0
1,NPC,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NPC ESPECIAL,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BARTIRA MASSA,96,1,0,326,1,5,BARTIRA MASSA,397191.41127,0.0,0.0,0.0,0.0,4266.210366,401457.621636,0.0,1.375373e+07,0.0,173887.53,0.0,-79199.78,-235579.705049,0.0,264648.436128,1.387749e+07,0.0
4,BARTIRA ESPECIAL,0,0,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [0]:
import openpyxl
import shutil # Você já importou isso na célula 5

# --- 1. Defina seus DataFrames e Caminhos ---

df_para_copiar = df_resultado 
caminho_template = '/Volumes/databox/juridico_comum/arquivos/MAPAS_MOVIMENTAÇÃO/INPUT/TEMPLATE/Mapa de Movimentação Template.xlsx'

# --- CORREÇÃO: Definir dois caminhos ---
# 1. Um caminho local temporário (no "disco" do driver)
caminho_local_temp = '/tmp/Mapa_Preenchido_temp.xlsx' 
# 2. O caminho final no Volume do Databricks
caminho_saida_final = '/Volumes/databox/juridico_comum/arquivos/MAPAS_MOVIMENTAÇÃO/INPUT/TRABALHISTA/MAPA_FINAL/Mapa_Preenchido.xlsx'


# --- 2. Mapeamento das Colunas ---
colunas_mapa = {
    'Provável Sdo inicial' : 8,
    'Adição_x' : 9,
    'ENCERRADO_x' : 10,
    'Remoto Sdo inicial' : 18,
    'Adição_y' : 19,
    'ENCERRADO_y' : 20,
    # COMEÇA SOCIO
    'Saldo Inicial SÓCIO' : 23,
    'ADIÇÃO NOVO' : 24,
    'OUTRAS ADIÇÕES' : 25,
    'ENCERRADO' : 26,
    'OUTRAS REVERSÕES' : 28,
    'ATUALIZAÇÃO' : 29,
    # COMEÇA EMPRESA
    'Saldo Inicial EMPRESA' : 32,
    'ADIÇÃO NOVO.1' : 33,
    'OUTRAS ADIÇÕES.1' : 34,
    'ENCERRADO.1' : 35,
    'OUTRAS REVERSÕES.1' : 37,
    'ATUALIZAÇÃO.1' : 38
}

# --- 3. Processo com Openpyxl ---
try:
    print(f"Carregando template: {caminho_template}...")
    wb = openpyxl.load_workbook(caminho_template)
    ws = wb['Mapa de Movimentação']
    print(f"Aba '{ws.title}' selecionada.")

    linha_inicial = 9 

    # Loop para colar os dados (seu código original está correto)
    for nome_coluna_df, numero_coluna_excel in colunas_mapa.items():
        if nome_coluna_df in df_para_copiar.columns:
            valores_para_colar = df_para_copiar[nome_coluna_df]
            print(f"Colando dados da coluna '{nome_coluna_df}' no Excel (Coluna {numero_coluna_excel})...")
            for i, valor in enumerate(valores_para_colar):
                linha_atual = linha_inicial + i
                ws.cell(row=linha_atual, column=numero_coluna_excel, value=valor)
        else:
            print(f"AVISO: Coluna '{nome_coluna_df}' não encontrada no DataFrame. Pulando.")

    # --- CORREÇÃO NA FORMA DE SALVAR ---
    
    # 1. Salva no arquivo local temporário
    print(f"Salvando arquivo temporário em: {caminho_local_temp}...")
    wb.save(caminho_local_temp)
    
    # 2. Move o arquivo do local temporário para o Volume final
    print(f"Movendo arquivo para o destino final: {caminho_saida_final}...")
    shutil.move(caminho_local_temp, caminho_saida_final)
    
    print("Sucesso! O template foi preenchido e salvo no Databricks.")

except FileNotFoundError:
    print(f"ERRO: O arquivo de template não foi encontrado em: {caminho_template}")
except KeyError:
    print(f"ERRO: A aba 'Mapa de Movimentação' não foi encontrada no template.")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

Carregando template: /Volumes/databox/juridico_comum/arquivos/MAPAS_MOVIMENTAÇÃO/INPUT/TEMPLATE/Mapa de Movimentação Template.xlsx...
Aba 'Mapa de Movimentação' selecionada.
Colando dados da coluna 'Provável Sdo inicial' no Excel (Coluna 8)...
Colando dados da coluna 'Adição_x' no Excel (Coluna 9)...
Colando dados da coluna 'ENCERRADO_x' no Excel (Coluna 10)...
Colando dados da coluna 'Remoto Sdo inicial' no Excel (Coluna 18)...
Colando dados da coluna 'Adição_y' no Excel (Coluna 19)...
Colando dados da coluna 'ENCERRADO_y' no Excel (Coluna 20)...
Colando dados da coluna 'Saldo Inicial SÓCIO' no Excel (Coluna 23)...
Colando dados da coluna 'ADIÇÃO NOVO' no Excel (Coluna 24)...
Colando dados da coluna 'OUTRAS ADIÇÕES' no Excel (Coluna 25)...
Colando dados da coluna 'ENCERRADO' no Excel (Coluna 26)...
Colando dados da coluna 'OUTRAS REVERSÕES' no Excel (Coluna 28)...
Colando dados da coluna 'ATUALIZAÇÃO' no Excel (Coluna 29)...
Colando dados da coluna 'Saldo Inicial EMPRESA' no Excel (Co